In [1]:
"""
FRED Loader — Demo Script
==========================

Demonstrates every public API feature: category pulls, scoring,
macro assumption overrides, resampling, custom series, and
inspecting scored output.
"""

import sys, os

sys.path.insert(0, os.path.abspath(".."))

from load import pull_fred, Config
from load import (
    INFLATION,
    OUTPUT,
    LABOR,
    RATES,
    MONEY,
    HOUSING,
    CONSUMER,
    TRADE,
    FINANCIAL,
    FISCAL,
    INEQUALITY,
    SEP,
    ALL_SERIES,
    CATEGORIES,
)
from macro_scores import list_scored_columns
import pandas as pd

OUT = "example_outputs"

# ═════════════════════════════════════════════════════════════════════════
# 0 — Catalog overview
# ═════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("CATALOG OVERVIEW")
print("=" * 60)
print(f"\n{len(ALL_SERIES)} total series across {len(CATEGORIES)} categories:\n")
for name, cat in CATEGORIES.items():
    series_names = [friendly for friendly, _ in cat.values()]
    print(f"  {name:<12s} ({len(cat):>2d})  {', '.join(series_names[:3])}...")
print()

CATALOG OVERVIEW

62 total series across 12 categories:

  INFLATION    ( 9)  Headline_CPI, Core_CPI, PCE_Price_Index...
  OUTPUT       ( 5)  Nominal_GDP, Real_GDP, Real_GDP_Per_Capita...
  LABOR        ( 8)  Unemployment_Rate, U6_Underemployment, Nonfarm_Payrolls...
  RATES        (10)  FedFunds_Rate, FedFunds_Daily, Treasury_2Y...
  MONEY        ( 5)  M2_Money_Stock, Total_Reserves, Commercial_Loans...
  HOUSING      ( 5)  Case_Shiller_Home_Price, Housing_Starts, Building_Permits...
  CONSUMER     ( 4)  UMich_Consumer_Sentiment, Retail_Sales, Personal_Consumption_Expenditures...
  TRADE        ( 2)  Trade_Balance, Trade_Weighted_USD_Broad...
  FINANCIAL    ( 4)  VIX, SP500, Chicago_Fed_Financial_Conditions...
  FISCAL       ( 3)  Federal_Debt_Total, Federal_Surplus_Deficit, Monthly_Treasury_Statement_Deficit...
  INEQUALITY   ( 1)  Gini_Index_USA...
  SEP          ( 6)  SEP_FedFunds_Median, SEP_FedFunds_Median_LongerRun, SEP_RealGDP_Median...



In [2]:
# ═════════════════════════════════════════════════════════════════════════
# 1 — Basic category pulls (no scoring)
# ═════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("1 — BASIC CATEGORY PULLS")
print("=" * 60)

# Single category
config = Config(
    filename="inequality.csv",
    output_path=OUT,
    start="1960-01-01",
    series={**INEQUALITY},
)
df = pull_fred(config=config, apply_scores=False)
print(f"\nInequality shape: {df.shape}")
print(df.head())

# Multiple categories merged
config = Config(
    filename="rates_and_housing.csv",
    output_path=OUT,
    start="2000-01-01",
    series={**RATES, **HOUSING},
)
df = pull_fred(config=config, apply_scores=False)
print(f"\nRates + Housing shape: {df.shape}")
print(df.tail())

1 — BASIC CATEGORY PULLS
  ✓ Gini_Index_USA                           (SIPOVGINIUSA       A → last)

Loaded 1 series  |  3132 weeks  |  1 columns

Saved → example_outputs/inequality.csv

Inequality shape: (3132, 1)
            Gini_Index_USA
date                      
1963-01-04            36.7
1963-01-11            36.7
1963-01-18            36.7
1963-01-25            36.7
1963-02-01            36.7
  ✓ FedFunds_Rate                            (FEDFUNDS           M → last)
  ✓ FedFunds_Daily                           (DFF                D → mean)
  ✓ Treasury_2Y                              (DGS2               D → mean)
  ✓ Treasury_10Y                             (DGS10              D → mean)
  ✓ Treasury_30Y                             (DGS30              D → mean)
  ✓ Yield_Curve_10Y_2Y                       (T10Y2Y             D → mean)
  ✓ Yield_Curve_10Y_3M                       (T10Y3M             D → mean)
  ✓ HY_OAS_Spread                            (BAMLH0A0HYM2       D → me

In [4]:
# ═════════════════════════════════════════════════════════════════════════
# 2 — Mix categories with individual FRED IDs
# ═════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("2 — MIX CATEGORIES WITH ONE-OFF SERIES")
print("=" * 60)

custom_series = {
    **INFLATION,
    "UNRATE": ("Unemployment_Rate", "M"),
    "FEDFUNDS": ("FedFunds_Rate", "M"),
}

config = Config(
    filename="inflation_plus_policy.csv",
    output_path=OUT,
    start="2000-01-01",
    series=custom_series,
)
df = pull_fred(config=config, apply_scores=False)
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")
print(df.tail())


2 — MIX CATEGORIES WITH ONE-OFF SERIES
  ✓ Headline_CPI                             (CPIAUCSL           M → last)
  ✓ Core_CPI                                 (CPILFESL           M → last)
  ✓ PCE_Price_Index                          (PCEPI              M → last)
  ✓ Core_PCE                                 (PCEPILFE           M → last)
  ✓ CPI_Food                                 (CPIUFDSL           M → last)
  ✓ CPI_Energy                               (CPIENGSL           M → last)
  ✓ UMich_Inflation_Expectations             (MICH               M → last)
  ✓ Breakeven_Inflation_5Y                   (T5YIE              D → mean)
  ✓ Breakeven_Inflation_10Y                  (T10YIE             D → mean)
  ✓ Unemployment_Rate                        (UNRATE             M → last)
  ✓ FedFunds_Rate                            (FEDFUNDS           M → last)

Loaded 11 series  |  1370 weeks  |  11 columns

Saved → example_outputs/inflation_plus_policy.csv

Columns (11): ['Headline_CPI', 'Cor

In [14]:
# ═════════════════════════════════════════════════════════════════════════
# 3 — Full dataset with scoring
# ═════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("3 — FULL DATASET WITH SCORING")
print("=" * 60)

config = Config(
    filename="scored_full.csv",
    output_path=OUT,
    start="2007-01-01",
    series={**ALL_SERIES},
)
scored = pull_fred(config=config, apply_scores=True)
print(f"\nShape: {scored.shape}")
print(f"Raw series: {len(ALL_SERIES)} | Scored columns added: {scored.shape[1] - len(ALL_SERIES) - 1}")

# List every scored column by category
catalog = list_scored_columns()
for category, cols in catalog.items():
    print(f"\n── {category.upper()} ({len(cols)} columns) ──")
    for c in cols:
        print(f"    {c}")


3 — FULL DATASET WITH SCORING
  ✓ Headline_CPI                             (CPIAUCSL           M → last)
  ✓ Core_CPI                                 (CPILFESL           M → last)
  ✓ PCE_Price_Index                          (PCEPI              M → last)
  ✓ Core_PCE                                 (PCEPILFE           M → last)
  ✓ CPI_Food                                 (CPIUFDSL           M → last)
  ✓ CPI_Energy                               (CPIENGSL           M → last)
  ✓ UMich_Inflation_Expectations             (MICH               M → last)
  ✓ Breakeven_Inflation_5Y                   (T5YIE              D → mean)
  ✓ Breakeven_Inflation_10Y                  (T10YIE             D → mean)
  ✓ Nominal_GDP                              (GDP                Q → last)
  ✓ Real_GDP                                 (GDPC1              Q → last)
  ✓ Real_GDP_Per_Capita                      (A939RX0Q048SBEA    Q → last)
  ✓ Industrial_Production                    (INDPRO             M → 

In [6]:
# ═════════════════════════════════════════════════════════════════════════
# 4 — Inspect key scored columns
# ═════════════════════════════════════════════════════════════════════════
scored = pd.read_csv("example_outputs/scored_full.csv")
print("\n" + "=" * 60)
print("4 — INSPECT SCORED OUTPUT")
print("=" * 60)

# --- Regime flags ---
flag_cols = [c for c in scored.columns if c.startswith("Flag_")]
print(f"\n{len(flag_cols)} regime flags — most recent states:")
print(scored[["date"] + flag_cols].tail(5).to_string(index=False))

# --- Taylor Rule gap ---
print("\n\nTaylor Rule (positive gap = policy too loose):")
taylor = scored[["date", "FedFunds_Rate", "Taylor_Prescribed", "Taylor_Gap"]].dropna()
print(taylor.tail(10).to_string(index=False))

# --- Dual mandate tension ---
print("\n\nDual Mandate Tension:")
print("  Mandate_Regime:  3=stagflation, 1=tighten, 0=balanced, -1=ease, -3=overheating")
mandate = scored[["date", "Inflation_Pressure", "Employment_Pressure", "Mandate_Tension", "Mandate_Regime"]].dropna()
print(mandate.tail(10).to_string(index=False))

# --- Sahm Rule ---
print("\n\nSahm Rule Recession Indicator (triggered ≥ 0.50):")
sahm = scored[["date", "Unemployment_Rate", "Sahm_Indicator", "Flag_Sahm_Triggered"]].dropna()
print(sahm.tail(10).to_string(index=False))

# --- Yield curve ---
print("\n\nYield Curve Inversion:")
curve = scored[["date", "Yield_Curve_10Y_2Y", "Flag_Curve_Inverted_10Y2Y", "Yield_Curve_10Y_3M", "Flag_Curve_Inverted_10Y3M"]].dropna()
print(curve.tail(10).to_string(index=False))

# --- Expectations anchoring ---
print("\n\nInflation Expectations Anchoring (higher = less anchored):")
expect = scored[["date", "UMich_Inflation_Expectations", "Breakeven_Inflation_5Y", "Expect_Anchor_Score", "Flag_Deanchor_Mild", "Flag_Deanchor_Severe"]].dropna()
print(expect.tail(10).to_string(index=False))

# --- Financial conditions transmission ---
print("\n\nFCI Transmission (positive = policy transmitting normally):")
fci = scored[["date", "Policy_Stance_SEP", "Chicago_Fed_Financial_Conditions", "FCI_Transmission", "Flag_FCI_Contradiction"]].dropna()
print(fci.tail(10).to_string(index=False))


4 — INSPECT SCORED OUTPUT

16 regime flags — most recent states:
      date  Flag_Curve_Inverted_10Y2Y  Flag_Curve_Inverted_10Y3M  Flag_Sahm_Triggered  Flag_Taylor_Below_100bp  Flag_Taylor_Above_100bp  Flag_FCI_Contradiction  Flag_Deanchor_Mild  Flag_Deanchor_Severe  Flag_Deanchor_Hot  Flag_Deanchor_Cold  Flag_M2_Excess  Flag_M2_Contraction  Flag_Labor_Divergence  Flag_Neg_Real_Rate_HighInflation  Flag_HY_Stress  Flag_Fiscal_Dominance_Risk
2027-12-10                          0                          0                    0                        0                        1                       1                   1                     1                  1                   0               0                    1                      0                                 0               0                           0
2027-12-17                          0                          0                    0                        0                        1                       1                 

In [14]:
# ═════════════════════════════════════════════════════════════════════════
# 5 — Override macro assumptions (pi_star, u_star)
# ═════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("5 — SENSITIVITY: OVERRIDE MACRO ASSUMPTIONS")
print("=" * 60)

series_subset = {**SEP, **INFLATION, **LABOR, **RATES}  # Just a few series needed for Taylor + SEP scoring
# Default: pi_star=2.0, u_star=4.0
config_default = Config(
    filename="scored_default.csv",
    output_path=OUT,
    start="2015-01-01",
    series=series_subset,
)
df_default = pull_fred(config=config_default, apply_scores=True)

# Hawkish: lower target, lower NAIRU
config_hawkish = Config(
    filename="scored_hawkish.csv",
    output_path=OUT,
    start="2015-01-01",
    series=series_subset,
    pi_star=1.5,
    u_star=3.5,
)
df_hawkish = pull_fred(config=config_hawkish, apply_scores=True)

# Dovish: higher target, higher NAIRU
config_dovish = Config(
    filename="scored_dovish.csv",
    output_path=OUT,
    start="2015-01-01",
    series=series_subset,
    pi_star=2.5,
    u_star=4.5,
)
df_dovish = pull_fred(config=config_dovish, apply_scores=True)

comparison = pd.DataFrame({
    "date": df_default["date"],
    "Actual_FFR": df_default["FedFunds_Rate"],
    "Taylor_Default": df_default["Taylor_Prescribed"],
    "Taylor_Hawkish": df_hawkish["Taylor_Prescribed"],
    "Taylor_Dovish": df_dovish["Taylor_Prescribed"],
}).dropna()

print("\nTaylor Rule under different assumptions:")
print(f"  Default:  π*=2.0, u*=4.0")
print(f"  Hawkish:  π*=1.5, u*=3.5")
print(f"  Dovish:   π*=2.5, u*=4.5\n")
print(comparison.tail(15).to_string(index=False))


5 — SENSITIVITY: OVERRIDE MACRO ASSUMPTIONS
  ✓ SEP_FedFunds_Median                      (FEDTARMD           SEP → last)
  ✓ SEP_FedFunds_Median_LongerRun            (FEDTARMDLR         SEP → last)
  ✓ SEP_RealGDP_Median                       (GDPC1MD            SEP → last)
  ✓ SEP_Unemployment_Median                  (UNRATEMD           SEP → last)
  ✓ SEP_PCE_Inflation_Median                 (PCECTPIMD          SEP → last)
  ✓ SEP_CorePCE_Median                       (JCXFEMD            SEP → last)
  ✓ Headline_CPI                             (CPIAUCSL           M → last)
  ✓ Core_CPI                                 (CPILFESL           M → last)
  ✓ PCE_Price_Index                          (PCEPI              M → last)
  ✓ Core_PCE                                 (PCEPILFE           M → last)
  ✓ CPI_Food                                 (CPIUFDSL           M → last)
  ✓ CPI_Energy                               (CPIENGSL           M → last)
  ✓ UMich_Inflation_Expectations           

In [9]:
# ═════════════════════════════════════════════════════════════════════════
# 6 — Change resample frequency
# ═════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("6 — RESAMPLE FREQUENCY")
print("=" * 60)

# Monthly
config_monthly = Config(
    filename="monthly_labor.csv",
    output_path=OUT,
    start="2020-01-01",
    resample_rule="MS",
    series={**LABOR},
)
df_monthly = pull_fred(config=config_monthly, apply_scores=False)
print(f"\nMonthly resampling (MS):  {len(df_monthly)} rows")
print(df_monthly.head(10).to_string(index=False))

# Quarterly
config_quarterly = Config(
    filename="quarterly_output.csv",
    output_path=OUT,
    start="2010-01-01",
    resample_rule="QS",
    series={**OUTPUT},
)
df_quarterly = pull_fred(config=config_quarterly, apply_scores=False)
print(f"\nQuarterly resampling (QS):  {len(df_quarterly)} rows")
print(df_quarterly.head(10).to_string(index=False))


6 — RESAMPLE FREQUENCY
  ✓ Unemployment_Rate                        (UNRATE             M → last)
  ✓ U6_Underemployment                       (U6RATE             M → last)
  ✓ Nonfarm_Payrolls                         (PAYEMS             M → last)
  ✓ Initial_Jobless_Claims                   (ICSA               W → last)
  ✓ Avg_Weekly_Hours                         (AWHAETP            M → last)
  ✓ Avg_Hourly_Earnings                      (CES0500000003      M → last)
  ✓ JOLTS_Job_Openings                       (JTSJOL             M → last)
  ✓ Labor_Force_Participation                (CIVPART            M → last)

Loaded 8 series  |  75 weeks  |  8 columns

Saved → example_outputs/monthly_labor.csv

Monthly resampling (MS):  75 rows
 Unemployment_Rate  U6_Underemployment  Nonfarm_Payrolls  Initial_Jobless_Claims  Avg_Weekly_Hours  Avg_Hourly_Earnings  JOLTS_Job_Openings  Labor_Force_Participation
               3.6                 7.0          152031.0                210000.0       

In [11]:
# ═════════════════════════════════════════════════════════════════════════
# 7 — Custom mean_freqs
# ═════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("7 — CUSTOM MEAN FREQUENCIES")
print("=" * 60)
print("Default: only daily (D) series are averaged into weekly buckets.")
print("Override: average both daily AND weekly series.\n")

config = Config(
    filename="mean_dw.csv",
    output_path=OUT,
    start="2020-01-01",
    mean_freqs={"D", "W"},
    series={**RATES, **FINANCIAL},
)
df = pull_fred(config=config, apply_scores=False)
print(f"Shape: {df.shape}")
print(df.head())


7 — CUSTOM MEAN FREQUENCIES
Default: only daily (D) series are averaged into weekly buckets.
Override: average both daily AND weekly series.

  ✓ FedFunds_Rate                            (FEDFUNDS           M → last)
  ✓ FedFunds_Daily                           (DFF                D → mean)
  ✓ Treasury_2Y                              (DGS2               D → mean)
  ✓ Treasury_10Y                             (DGS10              D → mean)
  ✓ Treasury_30Y                             (DGS30              D → mean)
  ✓ Yield_Curve_10Y_2Y                       (T10Y2Y             D → mean)
  ✓ Yield_Curve_10Y_3M                       (T10Y3M             D → mean)
  ✓ HY_OAS_Spread                            (BAMLH0A0HYM2       D → mean)
  ✓ IG_OAS_Spread                            (BAMLC0A0CM         D → mean)
  ✓ Mortgage_Rate_30Y                        (MORTGAGE30US       W → mean)
  ✓ VIX                                      (VIXCLS             D → mean)
  ✓ SP500                       

In [12]:
# ═════════════════════════════════════════════════════════════════════════
# 8 — Lead/lag predictive signals
# ═════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("8 — LEAD/LAG PREDICTIVE SIGNALS")
print("=" * 60)
print("_Lag_NNw  = value NN weeks AGO on current row (predictor)")
print("_Lead_NNw = value NN weeks in FUTURE on current row (target)\n")

# M2 → CPI (18-month lead)
print("M2 growth 18mo ago vs current CPI:")
m2_cpi = scored[["date", "M2_YoY", "M2_YoY_Lag_78w", "CPI_YoY"]].dropna()
print(m2_cpi.tail(10).to_string(index=False))

# Initial claims → unemployment (13-26 week lead)
print("\nInitial claims lagged vs unemployment:")
claims_ue = scored[["date", "Initial_Jobless_Claims", "Claims_Lag_13w", "Claims_Lag_26w", "Unemployment_Rate"]].dropna()
print(claims_ue.tail(10).to_string(index=False))

# Yield curve → recession signal (52 week lead)
print("\nYield curve 1 year ago vs current conditions:")
curve_lag = scored[["date", "Yield_Curve_10Y_2Y", "Curve_10Y2Y_Lag_52w", "Flag_Curve_Inverted_10Y2Y"]].dropna()
print(curve_lag.tail(10).to_string(index=False))

# Credit impulse → consumption
print("\nCredit impulse lagged (predicts consumption/activity):")
credit = scored[["date", "Credit_Impulse", "Credit_Impulse_Lag_13w", "Credit_Impulse_Lag_26w"]].dropna()
print(credit.tail(10).to_string(index=False))


8 — LEAD/LAG PREDICTIVE SIGNALS
_Lag_NNw  = value NN weeks AGO on current row (predictor)
_Lead_NNw = value NN weeks in FUTURE on current row (target)

M2 growth 18mo ago vs current CPI:
      date  M2_YoY  M2_YoY_Lag_78w  CPI_YoY
2027-11-05     0.0        3.817476      0.0
2027-11-12     0.0        3.817476      0.0
2027-11-19     0.0        3.817476      0.0
2027-11-26     0.0        3.817476      0.0
2027-12-03     0.0        3.321072      0.0
2027-12-10     0.0        3.321072      0.0
2027-12-17     0.0        3.321072      0.0
2027-12-24     0.0        3.321072      0.0
2027-12-31     0.0        2.940068      0.0
2028-01-07     0.0        2.940068      0.0

Initial claims lagged vs unemployment:
      date  Initial_Jobless_Claims  Claims_Lag_13w  Claims_Lag_26w  Unemployment_Rate
2027-11-05                202000.0        202000.0        202000.0                4.4
2027-11-12                202000.0        202000.0        202000.0                4.4
2027-11-19                2020

In [13]:
# ═════════════════════════════════════════════════════════════════════════
# 9 — Fiscal-monetary conflict & dominance risk
# ═════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("9 — FISCAL-MONETARY INTERACTION")
print("=" * 60)
print("Negative conflict score = fiscal policy offsetting monetary policy\n")

fiscal = scored[[
    "date", "Debt_to_GDP", "Deficit_GDP_Pct",
    "Policy_Stance_SEP", "Fiscal_Monetary_Conflict",
    "Flag_Fiscal_Dominance_Risk",
]].dropna()
print(fiscal.tail(15).to_string(index=False))


# ═════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("DEMO COMPLETE")
print("=" * 60)
print(f"\nAll outputs saved to: {OUT}/")
for f in sorted(os.listdir(OUT)):
    size = os.path.getsize(os.path.join(OUT, f))
    print(f"  {f:<45s} {size/1024:.0f} KB")


9 — FISCAL-MONETARY INTERACTION
Negative conflict score = fiscal policy offsetting monetary policy

      date  Debt_to_GDP  Deficit_GDP_Pct  Policy_Stance_SEP  Fiscal_Monetary_Conflict  Flag_Fiscal_Dominance_Risk
2027-10-01   122.490355       -11.735769               0.54                 -6.337315                           0
2027-10-08   122.490355       -11.735769               0.54                 -6.337315                           0
2027-10-15   122.490355       -11.735769               0.54                 -6.337315                           0
2027-10-22   122.490355       -11.735769               0.54                 -6.337315                           0
2027-10-29   122.490355       -11.735769               0.54                 -6.337315                           0
2027-11-05   122.490355       -11.735769               0.54                 -6.337315                           0
2027-11-12   122.490355       -11.735769               0.54                 -6.337315                